# PM2.5-Prognose (konfigurierbar): Split-Datum, Kovariaten-Auswahl, Chronos-2

Dies ist eine **parametrisierbare** Variante von `PM25_Chronos2_Forecast.ipynb` fuer
denselben Datensatz ["Beijing Multi-Site Air Quality Data"](https://archive.ics.uci.edu/dataset/501/beijing+multi+site+air+quality+data).

Anders als im Hauptnotebook sind hier direkt zentral einstellbar:

- **Split-Datum** (`SPLIT_DATE`) fuer Train/Test - statt eines fixen Testfensters in
  Tagen kannst du direkt ein Datum waehlen (z.B. um gezielt einen bestimmten Winter
  oder ein Ereignis als Testzeitraum zu pruefen).
- **Exogene Kovariaten** (`PAST_ONLY_COVARIATES` / `KNOWN_FUTURE_COVARIATES`) - frei
  waehlbar aus allen verfuegbaren Spalten, sauber getrennt in *past-only* (nur
  Historie bekannt, z.B. Schadstoffe) und *known-future* (Zukunftswerte bekannt/
  angenommen, z.B. Wetter, Kalender).
- **Feiertage** (`INCLUDE_HOLIDAYS`) - An/Aus-Schalter fuer chinesisches
  Neujahrsfest + sonstige Feiertage als Kovariaten-Kandidaten.

Der Prognosehorizont ist **fest auf 72 Stunden** gesetzt. Die Datenaufbereitung folgt
demselben Ansatz wie im Hauptnotebook: lineare NaN-Interpolation (kein Zeitlimit) +
Aggregation von Stunden- auf 3h-Aufloesung, damit der Chronos-2-Kontext (max. 8192
Zeitpunkte) mehrere Jahre Historie abdecken kann.

## 1. Setup

In [1]:
# Benoetigte Pakete installieren (einmalig; bei Bedarf auskommentieren)
# %pip install -q "chronos-forecasting>=2.0" "pandas[pyarrow]" matplotlib numpy scipy

import glob
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

RNG_SEED = 42
np.random.seed(RNG_SEED)

## 2. Konfiguration

Alle Stellschrauben dieses Notebooks stehen hier gesammelt. Nach Aenderungen einfach
das gesamte Notebook neu ausfuehren (Kernel-Neustart empfohlen, da praktisch alle
spaeteren Zellen von diesen Werten abhaengen).

In [2]:
# -----------------------------------------------------------------------
# Pfade
# -----------------------------------------------------------------------
DATA_DIR = r"C:\Users\betti\Desktop\Individual\beijing+multi+site+air+quality+data\PRSA2017_Data_20130301-20170228\PRSA_Data_20130301-20170228"
FILE_PATTERN = os.path.join(DATA_DIR, "PRSA_Data_*.csv")

TARGET_COL = "PM2.5"

# -----------------------------------------------------------------------
# Zeitaufloesung & Prognosehorizont
# -----------------------------------------------------------------------
AGG_RULE = "3h"     # Aggregation von Stunden- auf 3h-Aufloesung (Abschnitt 5)
STEP_HOURS = 3      # muss zu AGG_RULE passen

HORIZON_HOURS = 72   # fester Prognosehorizont (immer 72h)
assert HORIZON_HOURS % STEP_HOURS == 0, "HORIZON_HOURS muss durch STEP_HOURS teilbar sein"

# Chronos-2 hat laut Modellkarte eine maximale Kontextlaenge von 8192 Zeitpunkten.
# Laengere Historien wuerden von der Pipeline sonst stillschweigend gekuerzt -
# wir kuerzen daher explizit selbst (in Punkten, nicht Stunden).
MAX_CONTEXT_POINTS = 8192

# -----------------------------------------------------------------------
# Train/Test-Split: frei waehlbares Split-Datum statt fixer Testfenstergroesse
# -----------------------------------------------------------------------
SPLIT_DATE = "2016-06-01"  # alles davor = Train/Kontext, alles ab hier = Test
# Tipp: verschiedene Split-Datumsangaben ausprobieren, um Robustheit ueber
# unterschiedliche Zeitraeume/Jahreszeiten zu pruefen, z.B. "2015-01-01", "2016-01-01" ...

# -----------------------------------------------------------------------
# Feiertage optional
# -----------------------------------------------------------------------
INCLUDE_HOLIDAYS = True  # False -> Neujahrsfest-/Feiertags-Features werden gar nicht erst erzeugt

# -----------------------------------------------------------------------
# Konfidenzintervall
# -----------------------------------------------------------------------
CONF_LEVEL = 0.80
ALPHA = (1 - CONF_LEVEL) / 2
QUANTILE_LEVELS = [ALPHA, 0.5, 1 - ALPHA]
Q_LOW, Q_MED, Q_HIGH = [str(q) for q in QUANTILE_LEVELS]

CHRONOS_MODEL_ID = "amazon/chronos-2"

print(f"Horizont: {HORIZON_HOURS}h ({HORIZON_HOURS // STEP_HOURS} Zeitschritte a {STEP_HOURS}h)")
print(f"Split-Datum: {SPLIT_DATE}")
print(f"Feiertage inkludiert: {INCLUDE_HOLIDAYS}")

Horizont: 72h (24 Zeitschritte a 3h)
Split-Datum: 2016-06-01
Feiertage inkludiert: True


## 3. Daten laden

In [3]:
def load_all_stations(pattern: str) -> pd.DataFrame:
    '''Laedt alle PRSA_Data_*.csv-Dateien und haengt sie zu einem Long-Format-DataFrame zusammen.'''
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(
            f"Keine Dateien gefunden unter Muster: {pattern}. "
            "Bitte alle 12 PRSA_Data_*.csv-Dateien des UCI-Datensatzes in DATA_DIR ablegen."
        )
    frames = [pd.read_csv(f) for f in files]
    df_all = pd.concat(frames, ignore_index=True)

    df_all["timestamp"] = pd.to_datetime(df_all[["year", "month", "day", "hour"]])
    df_all = df_all.drop(columns=["No"], errors="ignore")
    df_all = df_all.sort_values(["station", "timestamp"]).reset_index(drop=True)
    return df_all


raw_df = load_all_stations(FILE_PATTERN)
stations = sorted(raw_df["station"].unique())
example_station = stations[0]
print(f"Gefundene Stationen ({len(stations)}): {stations}")
print(f"Zeitraum: {raw_df['timestamp'].min()} bis {raw_df['timestamp'].max()}")
raw_df.head()

FileNotFoundError: Keine Dateien gefunden unter Muster: C:\Users\betti\Desktop\Individual\beijing+multi+site+air+quality+data\PRSA2017_Data_20130301-20170228\PRSA_Data_20130301-20170228\PRSA_Data_*.csv. Bitte alle 12 PRSA_Data_*.csv-Dateien des UCI-Datensatzes in DATA_DIR ablegen.

## 4. Datenbereinigung

### 4.1 Fehlende Werte: lineare Interpolation
Alle numerischen Messspalten (inkl. Zielvariable) werden je Station **rein linear
interpoliert** (`interpolate(method="linear")`, kein Zeitlimit) - PM2.5 & Wetter sind
stundenweise stark autokorreliert. `ffill`/`bfill` dient nur noch als Fallback fuer
Werte ganz am Rand einer Stationszeitreihe, fuer die Interpolation keine zwei
Stuetzpunkte hat.

### 4.2 Windrichtung
`wd` ist eine 16-Punkte-Kompassrichtung (kategorial + zyklisch). Wir kodieren sie
sowohl zyklisch (`wd_sin`/`wd_cos`) als auch als Kategorie (`wd_cat`), damit sie
spaeter je nach Wahl numerisch oder als native Chronos-2-Kategorie genutzt werden kann.

In [ ]:
POLLUTANT_COLS = ["SO2", "NO2", "CO", "O3"]
WEATHER_NUMERIC_COLS = ["TEMP", "PRES", "DEWP", "RAIN", "WSPM"]
NUMERIC_COLS = [TARGET_COL, "PM10"] + POLLUTANT_COLS + WEATHER_NUMERIC_COLS


def clean_numeric_gaps(df: pd.DataFrame, cols=NUMERIC_COLS) -> pd.DataFrame:
    '''Fuellt NaNs in numerischen Spalten je Station: reine lineare Interpolation
    (kein Laengenlimit) + ffill/bfill als Fallback fuer Werte am Rand der Zeitreihe.'''
    df = df.copy()
    filled_parts = []
    for st, g in df.groupby("station", sort=False):
        g = g.sort_values("timestamp").set_index("timestamp")
        g[cols] = (
            g[cols]
            .interpolate(method="linear", limit_direction="both")
            .ffill()
            .bfill()
        )
        filled_parts.append(g.reset_index())
    return pd.concat(filled_parts, ignore_index=True).sort_values(["station", "timestamp"]).reset_index(drop=True)


n_missing_before = raw_df[NUMERIC_COLS].isna().sum().sum()
df_clean = clean_numeric_gaps(raw_df)
print(f"Fehlende numerische Werte vorher: {n_missing_before}, nachher: {df_clean[NUMERIC_COLS].isna().sum().sum()}")

In [ ]:
COMPASS_16 = ["N","NNE","NE","ENE","E","ESE","SE","SSE","S","SSW","SW","WSW","W","WNW","NW","NNW"]
COMPASS_DEGREES = {d: i * 22.5 for i, d in enumerate(COMPASS_16)}


def encode_wind_direction(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    deg = df["wd"].map(COMPASS_DEGREES)
    rad = np.deg2rad(deg)
    df["wd_sin"] = np.sin(rad)
    df["wd_cos"] = np.cos(rad)
    df["wd_sin"] = df.groupby("station")["wd_sin"].transform(lambda s: s.ffill().bfill())
    df["wd_cos"] = df.groupby("station")["wd_cos"].transform(lambda s: s.ffill().bfill())
    df["wd_cat"] = df["wd"].fillna("missing").astype("category")
    return df


df_clean = encode_wind_direction(df_clean)
df_clean[["wd", "wd_cat", "wd_sin", "wd_cos"]].head()

## 5. Aggregation auf 3h-Aufloesung

Um saisonale Effekte (volle Jahreszyklen) trotz der Chronos-2-Kontextgrenze von 8192
Punkten im Kontext sichtbar zu halten, aggregieren wir alle Groessen von Stunden- auf
**3-Stunden-Aufloesung** (`AGG_RULE`):

- **Mittelwert** fuer Zielvariable, Schadstoffe, Wettergroessen (TEMP, PRES, DEWP, WSPM)
  sowie `wd_sin`/`wd_cos`.
- **Summe** fuer `RAIN` (Niederschlag akkumuliert sich ueber das Zeitfenster).
- **Windrichtung**: `wd_cat` wird zirkulaer aus dem gemittelten `wd_sin`/`wd_cos`-Vektor
  zurueckgerechnet (naechstgelegene der 16 Kompassrichtungen) statt per Modus bestimmt.

Bei `MAX_CONTEXT_POINTS = 8192` deckt der Kontext dadurch ~2,8 Jahre statt ~341 Tage ab.
Der Horizont `HORIZON_HOURS = 72` ist durch `STEP_HOURS = 3` teilbar und passt daher
exakt auf das neue Raster.

In [ ]:
MEAN_COLS = [TARGET_COL, "PM10"] + POLLUTANT_COLS + ["TEMP", "PRES", "DEWP", "WSPM", "wd_sin", "wd_cos"]
SUM_COLS = ["RAIN"]


def aggregate_to_resolution(df: pd.DataFrame, rule: str = AGG_RULE) -> pd.DataFrame:
    '''Aggregiert alle Stationen von Stunden- auf `rule`-Aufloesung (Standard: 3h).'''
    parts = []
    for st, g in df.groupby("station", sort=False):
        g = g.set_index("timestamp").sort_index()
        agg = pd.concat([
            g[MEAN_COLS].resample(rule).mean(),
            g[SUM_COLS].resample(rule).sum(),
        ], axis=1)
        agg["station"] = st
        parts.append(agg.reset_index())

    df_agg = pd.concat(parts, ignore_index=True)

    deg = np.degrees(np.arctan2(df_agg["wd_sin"], df_agg["wd_cos"])) % 360
    compass_idx = (deg / 22.5).round().astype(int) % 16
    df_agg["wd_cat"] = pd.Categorical([COMPASS_16[i] for i in compass_idx], categories=COMPASS_16)

    return df_agg.sort_values(["station", "timestamp"]).reset_index(drop=True)


n_rows_before_agg = len(df_clean)
df_clean = aggregate_to_resolution(df_clean)
print(f"Zeilen vor Aggregation (stuendlich): {n_rows_before_agg}")
print(f"Zeilen nach Aggregation auf {AGG_RULE}:  {len(df_clean)}")
df_clean[["station", "timestamp", TARGET_COL, "wd_cat", "wd_sin", "wd_cos", "RAIN"]].head()

## 6. Kalender-Features (+ optional Feiertage)

`hour`, `dow`, `month`, `is_weekend` werden immer berechnet. Das chinesische
Neujahrsfest (`days_to_cny`, `cny_window`) sowie sonstige Feiertage
(`is_public_holiday`) werden **nur** berechnet, wenn `INCLUDE_HOLIDAYS = True`
(Konfiguration, Abschnitt 2) - bei `False` existieren diese Spalten gar nicht erst und
stehen bei der Kovariatenauswahl (Abschnitt 7) folglich auch nicht als Kandidaten zur
Verfuegung.

In [ ]:
CNY_DATES = pd.to_datetime(list({
    2013: "2013-02-10", 2014: "2014-01-31", 2015: "2015-02-19",
    2016: "2016-02-08", 2017: "2017-01-28",
    2012: "2012-01-23", 2018: "2018-02-16",  # Randjahre fuer saubere "naechstgelegen"-Berechnung
}.values()))

OTHER_HOLIDAYS = [
    ("2013-01-01", "2013-01-01"), ("2013-04-29", "2013-05-01"), ("2013-10-01", "2013-10-07"),
    ("2014-01-01", "2014-01-01"), ("2014-05-01", "2014-05-01"), ("2014-10-01", "2014-10-07"),
    ("2015-01-01", "2015-01-01"), ("2015-05-01", "2015-05-03"), ("2015-10-01", "2015-10-07"),
    ("2016-01-01", "2016-01-03"), ("2016-05-01", "2016-05-02"), ("2016-10-01", "2016-10-07"),
    ("2017-01-01", "2017-01-02"), ("2017-05-01", "2017-05-01"),
]
OTHER_HOLIDAYS = [(pd.Timestamp(a), pd.Timestamp(b)) for a, b in OTHER_HOLIDAYS]
CNY_WINDOW_DAYS = 7


def add_calendar_and_holiday_features(df: pd.DataFrame, include_holidays: bool = INCLUDE_HOLIDAYS) -> pd.DataFrame:
    df = df.copy()
    ts = df["timestamp"]

    df["hour"] = ts.dt.hour
    df["dow"] = ts.dt.dayofweek
    df["month"] = ts.dt.month
    df["is_weekend"] = (df["dow"] >= 5).astype(int)

    if include_holidays:
        dates_only = ts.dt.normalize()
        diffs = np.subtract.outer(dates_only.values.astype("datetime64[D]"), CNY_DATES.values.astype("datetime64[D]"))
        diffs_days = diffs / np.timedelta64(1, "D")
        idx_min = np.abs(diffs_days).argmin(axis=1)
        df["days_to_cny"] = diffs_days[np.arange(len(df)), idx_min].astype(int)
        df["cny_window"] = (df["days_to_cny"].abs() <= CNY_WINDOW_DAYS).astype(int)

        is_holiday = pd.Series(False, index=df.index)
        for start, end in OTHER_HOLIDAYS:
            is_holiday |= dates_only.between(start, end)
        df["is_public_holiday"] = (is_holiday | (df["cny_window"] == 1)).astype(int)

    return df


df_clean = add_calendar_and_holiday_features(df_clean)
holiday_cols = ["days_to_cny", "cny_window", "is_public_holiday"] if INCLUDE_HOLIDAYS else []
df_clean[["timestamp", "hour", "dow", "is_weekend"] + holiday_cols].sample(5, random_state=RNG_SEED)

## 7. Exogene Kovariaten auswaehlen (past-only vs. known-future)

Waehle hier explizit, welche Spalten als **past-only** (nur Historie bekannt, z.B.
Schadstoffe - deren Zukunft man zum echten Prognosezeitpunkt nicht kennt) und welche
als **known-future** (Zukunftswerte bekannt/angenommen, z.B. Wettervorhersage oder
Kalenderfeatures) an Chronos-2 uebergeben werden. Chronos-2 erkennt eine Kovariate
als "past-only", wenn sie im Kontext, aber nicht in den Zukunftsdaten vorkommt.

**Verfuegbare Kandidaten:**

- Past-only: `SO2, NO2, CO, O3`
- Known-future (numerisch): `TEMP, PRES, DEWP, RAIN, WSPM, wd_sin, wd_cos, hour, dow, month`
  (+ `days_to_cny`, falls `INCLUDE_HOLIDAYS = True`)
- Known-future (kategorial): `wd_cat, is_weekend`
  (+ `cny_window, is_public_holiday`, falls `INCLUDE_HOLIDAYS = True`)

Passe `PAST_ONLY_COVARIATES` und `KNOWN_FUTURE_COVARIATES` unten frei an (Teilmengen
der Kandidaten, disjunkt) - die Validierungszelle prueft automatisch, ob die Auswahl
gueltig ist.

In [ ]:
CANDIDATE_PAST_ONLY = ["SO2", "NO2", "CO", "O3"]

CANDIDATE_KNOWN_FUTURE_NUMERIC = ["TEMP", "PRES", "DEWP", "RAIN", "WSPM", "wd_sin", "wd_cos", "hour", "dow", "month"]
CANDIDATE_KNOWN_FUTURE_CATEGORICAL = ["wd_cat", "is_weekend"]
if INCLUDE_HOLIDAYS:
    CANDIDATE_KNOWN_FUTURE_NUMERIC = CANDIDATE_KNOWN_FUTURE_NUMERIC + ["days_to_cny"]
    CANDIDATE_KNOWN_FUTURE_CATEGORICAL = CANDIDATE_KNOWN_FUTURE_CATEGORICAL + ["cny_window", "is_public_holiday"]
CANDIDATE_KNOWN_FUTURE = CANDIDATE_KNOWN_FUTURE_NUMERIC + CANDIDATE_KNOWN_FUTURE_CATEGORICAL

# -----------------------------------------------------------------------
# HIER AUSWAEHLEN: frei editierbare Teilmengen der obigen Kandidaten
# -----------------------------------------------------------------------
PAST_ONLY_COVARIATES = ["SO2", "NO2", "CO", "O3"]

KNOWN_FUTURE_COVARIATES = [
    "TEMP", "DEWP", "WSPM", "wd_cat", "hour", "dow", "is_weekend",
] + (["cny_window"] if INCLUDE_HOLIDAYS else [])

# -----------------------------------------------------------------------
# Validierung
# -----------------------------------------------------------------------
assert set(PAST_ONLY_COVARIATES) <= set(CANDIDATE_PAST_ONLY), \
    f"Ungueltige past-only Kovariate(n): {set(PAST_ONLY_COVARIATES) - set(CANDIDATE_PAST_ONLY)}"
assert set(KNOWN_FUTURE_COVARIATES) <= set(CANDIDATE_KNOWN_FUTURE), \
    f"Ungueltige known-future Kovariate(n): {set(KNOWN_FUTURE_COVARIATES) - set(CANDIDATE_KNOWN_FUTURE)}"
assert set(PAST_ONLY_COVARIATES).isdisjoint(KNOWN_FUTURE_COVARIATES), \
    "Eine Kovariate darf nicht gleichzeitig past-only UND known-future sein"

print("Past-only Kovariaten:   ", PAST_ONLY_COVARIATES)
print("Known-future Kovariaten:", KNOWN_FUTURE_COVARIATES)

## 8. Train/Test-Split nach Split-Datum

`SPLIT_DATE` (Konfiguration, Abschnitt 2) trennt den Trainings-/Kontextzeitraum
(alles davor) vom Testzeitraum (alles ab diesem Datum bis zum Datenende). Anders als
ein fixes Testfenster in Tagen laesst sich so gezielt z.B. ein bestimmter Winter/
Sommer oder ein Ereigniszeitraum als Testfenster waehlen - einfach `SPLIT_DATE` in
Abschnitt 2 aendern und das Notebook neu ausfuehren. Der Split erfolgt je Station
identisch (gleicher Zeitpunkt).

In [ ]:
def train_test_split_by_date(df: pd.DataFrame, split_date: str):
    split_ts = pd.Timestamp(split_date)
    train_df = df.loc[df["timestamp"] < split_ts].copy()
    test_df = df.loc[df["timestamp"] >= split_ts].copy()
    return train_df, test_df, split_ts


train_df, test_df, TRAIN_END = train_test_split_by_date(df_clean, SPLIT_DATE)
print(f"Trainingszeitraum: {df_clean['timestamp'].min()} bis {TRAIN_END} ({len(train_df)} Zeilen)")
print(f"Testzeitraum:      {TRAIN_END} bis {df_clean['timestamp'].max()} ({len(test_df)} Zeilen)")

data_end = df_clean["timestamp"].max()
if len(test_df) == 0 or (data_end - TRAIN_END) < pd.Timedelta(hours=HORIZON_HOURS):
    raise ValueError(
        f"SPLIT_DATE={SPLIT_DATE!r} laesst keinen Test-/Backtestzeitraum von mindestens "
        f"{HORIZON_HOURS}h bis zum Datenende ({data_end}) - bitte eine fruehere SPLIT_DATE waehlen."
    )

## 9. Chronos-2 laden

`Chronos2Pipeline.from_pretrained(...)` laedt das vortrainierte Modell von Hugging
Face. `device_map` wird automatisch auf `"cuda"` gesetzt, falls eine GPU verfuegbar
ist, sonst auf `"cpu"`.

In [ ]:
import torch
from chronos import Chronos2Pipeline

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Verwende Device: {DEVICE}")

pipeline = Chronos2Pipeline.from_pretrained(CHRONOS_MODEL_ID, device_map=DEVICE)

## 10. Prognosefunktionen (fester 72h-Horizont)

`prediction_length` wird `pipeline.predict_df` in **Zeitschritten** (nicht Stunden)
uebergeben: `HORIZON_HOURS // STEP_HOURS`. Der Kontext wird je Station explizit auf
`MAX_CONTEXT_POINTS` gekuerzt (Chronos-2-Limit von 8192 Punkten), statt sich auf eine
stille interne Kuerzung der Pipeline zu verlassen. `PAST_ONLY_COVARIATES` erscheinen
nur im Kontext, `KNOWN_FUTURE_COVARIATES` in Kontext **und** Zukunftsdaten (Abschnitt 7).

In [ ]:
ID_COL = "station"
TS_COL = "timestamp"


def build_context_and_future(df_hist: pd.DataFrame, origin: pd.Timestamp):
    '''Baut context_df (Historie bis origin, gekuerzt auf MAX_CONTEXT_POINTS je Station)
    und future_df (naechste HORIZON_HOURS Stunden) im Long-Format fuer predict_df.'''
    context_cols = [ID_COL, TS_COL, TARGET_COL] + PAST_ONLY_COVARIATES + KNOWN_FUTURE_COVARIATES
    future_cols = [ID_COL, TS_COL] + KNOWN_FUTURE_COVARIATES

    context_df = df_hist.loc[df_hist[TS_COL] <= origin, context_cols].copy()
    context_df = (
        context_df.sort_values([ID_COL, TS_COL])
        .groupby(ID_COL, group_keys=False)
        .apply(lambda g: g.tail(MAX_CONTEXT_POINTS))
    )

    future_start = origin + pd.Timedelta(hours=STEP_HOURS)
    future_end = origin + pd.Timedelta(hours=HORIZON_HOURS)
    future_df = df_hist.loc[
        (df_hist[TS_COL] >= future_start) & (df_hist[TS_COL] <= future_end), future_cols
    ].copy()

    return context_df, future_df


def forecast_at_origin(df_hist: pd.DataFrame, origin: pd.Timestamp) -> pd.DataFrame:
    '''Erzeugt eine HORIZON_HOURS-Prognose ab origin, fuer alle Stationen gleichzeitig.'''
    context_df, future_df = build_context_and_future(df_hist, origin)
    return pipeline.predict_df(
        context_df,
        future_df=future_df,
        prediction_length=HORIZON_HOURS // STEP_HOURS,  # Anzahl Zeitschritte, nicht Stunden
        quantile_levels=QUANTILE_LEVELS,
        id_column=ID_COL,
        timestamp_column=TS_COL,
        target=TARGET_COL,
    )


# Funktionstest (nur Datenaufbereitung, kein Modellaufruf)
_ctx, _fut = build_context_and_future(df_clean, TRAIN_END)
print("context_df:", _ctx.shape, "future_df:", _fut.shape,
      f"(erwartet: {HORIZON_HOURS // STEP_HOURS} Zeilen je Station in future_df)")
print("Kontext je Station <=", MAX_CONTEXT_POINTS, "Punkte:",
      (_ctx.groupby(ID_COL).size() <= MAX_CONTEXT_POINTS).all())
assert set(PAST_ONLY_COVARIATES).isdisjoint(_fut.columns), "Past-only Kovariaten duerfen NICHT in future_df stehen!"
print("OK: past-only Kovariaten sind nicht in future_df enthalten (kein Leakage).")

In [ ]:
# Beispielhafte Einzelprognose (Ursprung = Split-Datum)
example_pred = forecast_at_origin(df_clean, TRAIN_END)
example_pred.head()

In [ ]:
def plot_forecast(df_hist, pred_df, origin, station, context_hours=7 * 24):
    ctx = df_hist.loc[
        (df_hist[ID_COL] == station) &
        (df_hist[TS_COL] > origin - pd.Timedelta(hours=context_hours)) &
        (df_hist[TS_COL] <= origin)
    ]
    truth = df_hist.loc[
        (df_hist[ID_COL] == station) &
        (df_hist[TS_COL] > origin) &
        (df_hist[TS_COL] <= origin + pd.Timedelta(hours=HORIZON_HOURS))
    ]
    pred = pred_df.loc[pred_df[ID_COL] == station].sort_values(TS_COL)

    plt.figure(figsize=(12, 4))
    plt.plot(ctx[TS_COL], ctx[TARGET_COL], label="Historie", color="tab:blue")
    plt.plot(truth[TS_COL], truth[TARGET_COL], label="Tatsaechlicher Wert", color="tab:green")
    plt.plot(pred[TS_COL], pred["predictions"], label="Chronos-2 Prognose (Median)", color="tab:purple")
    plt.fill_between(pred[TS_COL], pred[Q_LOW], pred[Q_HIGH], color="tab:purple", alpha=0.2,
                      label=f"{int(CONF_LEVEL*100)}%-Konfidenzintervall")
    plt.axvline(origin, color="black", linestyle="--", linewidth=0.8)
    plt.title(f"PM2.5-Prognose ({HORIZON_HOURS}h) ab {origin} - {station}")
    plt.ylabel("PM2.5 [µg/m³]")
    plt.legend()
    plt.tight_layout()
    plt.show()


plot_forecast(df_clean, example_pred, TRAIN_END, example_station)

## 11. Metriken: MAPE und RMSE

In [ ]:
EPS = 1e-3  # verhindert Division durch 0 im MAPE bei sehr niedrigen PM2.5-Werten


def mape(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), EPS))) * 100)


def rmse(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

## 12. Sliding-Window-Backtesting (72h-Horizont)

Der Prognoseursprung wird schrittweise (`BACKTEST_STEP`) durch den Testzeitraum (ab
`TRAIN_END` = `SPLIT_DATE`) geschoben. An jedem Ursprung wird eine 72h-Prognose
erzeugt und am Ende (t+72h) mit dem tatsaechlichen Wert verglichen.

In [ ]:
BACKTEST_STEP = pd.Timedelta(days=1)
QUICK_TEST_N_ORIGINS = 5  # fuer einen schnellen Probelauf; auf None setzen fuer volles Backtesting


def generate_backtest_origins(train_end: pd.Timestamp, data_end: pd.Timestamp,
                               horizon_hours: int = HORIZON_HOURS, step: pd.Timedelta = BACKTEST_STEP):
    origins = []
    o = train_end
    while o + pd.Timedelta(hours=horizon_hours) <= data_end:
        origins.append(o)
        o += step
    return origins


backtest_origins = generate_backtest_origins(TRAIN_END, df_clean[TS_COL].max())
print(f"Anzahl Backtest-Ursprungszeitpunkte (voller Testzeitraum, step={BACKTEST_STEP}): {len(backtest_origins)}")


def run_sliding_window_backtest(df_hist: pd.DataFrame, origins: list) -> pd.DataFrame:
    '''Fuehrt das Sliding-Window-Backtesting durch: eine Zeile je (origin, station), Vergleich bei t+HORIZON_HOURS.'''
    records = []
    for origin in origins:
        pred = forecast_at_origin(df_hist, origin)
        target_ts = origin + pd.Timedelta(hours=HORIZON_HOURS)
        pred_h = pred.loc[pred[TS_COL] == target_ts]
        truth_h = df_hist.loc[df_hist[TS_COL] == target_ts, [ID_COL, TARGET_COL]]

        merged = pred_h.merge(truth_h, on=ID_COL, suffixes=("_pred", "_true"))
        for _, row in merged.iterrows():
            records.append({
                "origin": origin,
                "station": row[ID_COL],
                "y_true": row[TARGET_COL],
                "y_pred": row["predictions"],
                "q_low": row[Q_LOW],
                "q_high": row[Q_HIGH],
                "within_ci": (row[TARGET_COL] >= row[Q_LOW]) and (row[TARGET_COL] <= row[Q_HIGH]),
            })
    return pd.DataFrame.from_records(records)


# --- Schneller Probelauf (wenige Ursprungszeitpunkte), zur Kontrolle vor dem vollen Backtesting ---
quick_origins = backtest_origins[:QUICK_TEST_N_ORIGINS] if QUICK_TEST_N_ORIGINS else backtest_origins
backtest_results = run_sliding_window_backtest(df_clean, quick_origins)
backtest_results.head()

## 13. Auswertung

In [ ]:
summary = pd.Series({
    "MAPE_%": mape(backtest_results["y_true"], backtest_results["y_pred"]),
    "RMSE": rmse(backtest_results["y_true"], backtest_results["y_pred"]),
    "CI_coverage_%": backtest_results["within_ci"].mean() * 100,  # sollte nahe CONF_LEVEL*100 liegen
    "n_obs": len(backtest_results),
})
print(f"Konfiguration: Horizont={HORIZON_HOURS}h, Split-Datum={SPLIT_DATE}, "
      f"Feiertage={INCLUDE_HOLIDAYS}")
print(f"Past-only Kovariaten:    {PAST_ONLY_COVARIATES}")
print(f"Known-future Kovariaten: {KNOWN_FUTURE_COVARIATES}")
summary

In [ ]:
# Auswertung je Station (fuer den gewaehlten Backtest-Umfang)
summary_by_station = backtest_results.groupby("station").apply(
    lambda g: pd.Series({
        "MAPE_%": mape(g["y_true"], g["y_pred"]),
        "RMSE": rmse(g["y_true"], g["y_pred"]),
        "CI_coverage_%": g["within_ci"].mean() * 100,
        "n_obs": len(g),
    })
).reset_index()
summary_by_station

## 14. Zusammenfassung

**Konfigurierbar** (Abschnitt 2 bzw. 7):
- `SPLIT_DATE` - Train/Test-Trennung frei nach Datum, statt fixem Testfenster.
- `PAST_ONLY_COVARIATES` / `KNOWN_FUTURE_COVARIATES` - frei waehlbare exogene
  Kovariaten aus den jeweiligen Kandidatenlisten, mit automatischer Validierung
  (gueltige Spalten, keine Ueberschneidung).
- `INCLUDE_HOLIDAYS` - An/Aus fuer Neujahrsfest-/Feiertagsfeatures als Kandidaten.

**Fest:**
- Prognosehorizont **72h**.
- Datenaufbereitung: lineare NaN-Interpolation (kein Zeitlimit), Aggregation auf
  3h-Aufloesung (`AGG_RULE`), Chronos-2-Kontext-Cap auf `MAX_CONTEXT_POINTS` (8192
  Zeitpunkte, ~2,8 Jahre bei 3h-Aufloesung).

**Verschiedene Konfigurationen vergleichen:** Abschnitt 2 (Split-Datum, Feiertage) bzw.
Abschnitt 7 (Kovariaten) aendern und das Notebook neu ausfuehren (Kernel-Neustart
empfohlen) - `summary` und `summary_by_station` am Ende zeigen die Ergebnisse fuer die
jeweils aktuelle Einstellung. Fuer belastbarere Metriken `QUICK_TEST_N_ORIGINS = None`
setzen, um ueber den gesamten Testzeitraum statt nur 5 Ursprungszeitpunkte zu backtesten.